# Primary Model: CNN

In [4]:
# imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio
import numpy as np

## Load Dataset

In [5]:
# login Hugging Face Hub
import os
from dotenv import load_dotenv

load_dotenv()  
HF_TOKEN = os.getenv("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN) 

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Try primary model with smaller dataset first

In [6]:
# load dataset
# ds = load_dataset("SleepyJesse/ai_music_tiny", split="train", cache_dir="./data")
ds = load_dataset("SleepyJesse/ai_music_tiny", split="train")

# Avoid torchcodec decoding in environments where FFmpeg/TorchCodec isn't configured.
ds = ds.cast_column("audio", Audio(decode=False))
ds

Dataset({
    features: ['audio', 'ai_generated', 'source'],
    num_rows: 1000
})

## CNN Architecture

In [7]:
class MusicCNN(nn.Module):

    def __init__(self):

        super().__init__()

        # Block 1
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        # Block 2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # Block 3
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)

        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 16 * 27, 256)
        self.fc2 = nn.Linear(256, 1)


    def forward(self, x):

        # Block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = torch.flatten(x, 1)

        x = self.dropout(F.relu(self.fc1(x)))

        x = torch.sigmoid(self.fc2(x))

        return x